## Импорты и настройка

In [1]:
from datasets import load_dataset
from data_structures import Example
from hierarchical_optimizer import HierarchicalOptimizer

/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/kateav

### Функции для подготовки данных

In [2]:
def squad_v2_to_examples(data):
    examples = []
    for item in data:
        input_text = (f'# Question: \n {item["question"]} \n'
                      f'# Context: \n {item["context"]} \n')
        expected_output = item['answers']['text'][0] if item['answers']['text'] else 'No answer'
        examples.append(Example(input_text=input_text, expected_output=expected_output))
    return examples

def get_squad_v2_data(train_num: int, val_num: int, test_num: int):
    ds_train = load_dataset('rajpurkar/squad_v2', split='train')
    ds_test = load_dataset('rajpurkar/squad_v2', split='validation')

    split = ds_train.train_test_split(test_size=0.1, seed=42)
    ds_train = split['train']
    ds_val = split['test']

    ds_train = ds_train.shuffle()
    ds_val = ds_val.shuffle()
    ds_test = ds_test.shuffle()

    train_split = ds_train.select(range(train_num))
    val_split = ds_val.select(range(val_num))
    test_split = ds_test.select(range(test_num))

    train_examples = squad_v2_to_examples(train_split)
    validation_examples = squad_v2_to_examples(val_split)
    test_examples = squad_v2_to_examples(test_split)

    return train_examples, validation_examples, test_examples

def data_fabric(dataset: str = 'squad_v2', train_num: int = 150, val_num: int = 100, test_num: int = 300):
    squad_v2_initial_prompt = """Answer the question based on the context. If there is no answer in the context then just return 'No answer'."""

    train_examples, validation_examples, test_examples = get_squad_v2_data(train_num, val_num, test_num)
    initial_prompt = squad_v2_initial_prompt
    
    return train_examples, validation_examples, test_examples, initial_prompt

## Подготовка датасета
Создаем простой датасет для демонстрации (задача классификации тональности)

In [3]:
LABEL_MAP = {0: "truth", 1: "lie"}

train_examples, validation_examples, test_examples, initial_prompt = data_fabric('squad_v2')

print("Dataset prepared:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Validation: {len(validation_examples)} examples")
print(f"  Test: {len(test_examples)} examples")

Dataset prepared:
  Train: 150 examples
  Validation: 100 examples
  Test: 300 examples


## Создание начального промпта

In [4]:
print("Initial prompt:")
print("-" * 60)
print(initial_prompt)
print("-" * 60)

Initial prompt:
------------------------------------------------------------
Answer the question based on the context. If there is no answer in the context then just return 'No answer'.
------------------------------------------------------------


## Инициализация оптимизатора

In [5]:
optimizer = HierarchicalOptimizer()

## Запуск оптимизации

In [6]:
best_node = optimizer.optimize(
    initial_prompt=initial_prompt,
    train_examples=train_examples,
    validation_examples=validation_examples,
    test_examples=test_examples,
    save_dir="./optimization_results",
)

Evaluating initial prompt...
[diag] initial node: prompt_id=7c8e9ca8bec6 len=108 chars
[diag] Prompt text: Answer the question based on the context. If there is no answer in the context then just return 'No answer'.
[diag] evaluate_node: node_id=475ae99a-29e3-4311-a823-cb8d2eee9a80 gen=0 source=initial split=validation stage=1 examples=100 seed_offset=0
[diag] evaluate_prompt: execute=True sample=True stage=1 incoming=100 will_use=100 weights={accuracy:0.3, token_f1:0.2} llm_calls_at_start=0
[diag] execute_prompt_batch: total=100 batch_size=50 n_batches=2 llm_calls_before=0
[diag]   batch 1/2: OK (50/100 done, llm_calls=1)
[diag]   batch 2/2: OK (100/100 done, llm_calls=2)
[diag] execute_prompt_batch done: llm_calls_delta=2 total_calls=2
[diag] evaluate_prompt execution done: llm_calls_for_execution=2
[diag]   example[0] expected='No answer' actual='No answer'
[diag]   example[1] expected='No answer' actual='No answer'
[diag]   example[2] expected='No answer' actual='Until June 1932'
[

## Анализ результатов

In [7]:
report = optimizer.get_optimization_report()
print('Optimization generations summary:')
for entry in report['optimization_log']:
    print(f"  Generation {entry['generation']}: time {entry['time']:.2f}s, best_score {entry['best_score']:.3f}")

local_stats = report['component_statistics']['local_optimizer']
print('Local optimizer summary:')
print(f"  Total iterations recorded: {local_stats.get('total_iterations', 0)}")
avg_it = local_stats.get('avg_iteration_time')
if avg_it is not None:
    print(f"  Avg iteration time: {avg_it:.2f}s")
else:
    print('  Avg iteration time: N/A')
print(f"  Total LLM calls attributed to local iterations: {local_stats.get('total_llm_calls_by_local', 0)}")
print('Per-iteration breakdown:')
for s in local_stats.get('iteration_stats', []):
    print(f"  Iter {s['iteration']}: time {s['time']:.2f}s, llm_calls {s['llm_calls']}")

Optimization generations summary:
  Generation 1: time 538.11s, best_score 0.503
  Generation 2: time 1523.55s, best_score 0.503
  Generation 3: time 0.03s, best_score 0.503
Local optimizer summary:
  Total iterations recorded: 10
  Avg iteration time: 178.46s
  Total LLM calls attributed to local iterations: 152
Per-iteration breakdown:
  Iter 1: time 213.09s, llm_calls 17
  Iter 2: time 322.36s, llm_calls 29
  Iter 1: time 172.41s, llm_calls 15
  Iter 2: time 78.37s, llm_calls 6
  Iter 1: time 218.02s, llm_calls 17
  Iter 2: time 212.17s, llm_calls 18
  Iter 1: time 246.62s, llm_calls 22
  Iter 2: time 321.53s, llm_calls 28
  Iter 1: time 0.02s, llm_calls 0
  Iter 2: time 0.01s, llm_calls 0


In [8]:
print("BEST PROMPT FOUND:")
print("=" * 80)
print(best_node.prompt_text)
print("=" * 80)
print(f"Score: {best_node.metrics.composite_score():.3f}")
print(f"Generation: {best_node.generation}")
print(f"Source: {best_node.source.value}")

BEST PROMPT FOUND:
Provide the answer based only on the factual information presented in the context. If the answer is not in the context, return 'No answer'.
Score: 0.503
Generation: 1
Source: local


In [9]:
metrics = best_node.metrics

print("METRICS:")
print(f"  Composite Score: {metrics.composite_score():.3f}")
print()

# Key 5 metrics
key_metrics = ["accuracy", "f1", "safety", "robustness", "efficiency"]
print("  Key Metrics:")
for name in key_metrics:
    value = metrics.metrics.get(name, None)
    weight = metrics.weights.get(name, 0.0)
    if value is not None:
        print(f"    {name:15s}: {value:.3f}  (weight={weight}, contribution={value * weight:.4f})")
    else:
        print(f"    {name:15s}: N/A   (stage 3 metric — run test eval to compute)")

# Other metrics (token_f1 etc.)
other = {k: v for k, v in sorted(metrics.metrics.items()) if k not in key_metrics}
if other:
    print("  Screening Metrics:")
    for name, value in other.items():
        weight = metrics.weights.get(name, 0.0)
        print(f"    {name:15s}: {value:.3f}  (weight={weight}, contribution={value * weight:.4f})")

# Test metrics if available
if "test_metrics" in best_node.metadata:
    tm = best_node.metadata["test_metrics"]
    print()
    print("  Test Set Metrics (Stage 3 — all 5 criteria):")
    for name in key_metrics:
        value = tm.get(name, None)
        if value is not None:
            print(f"    {name:15s}: {value:.3f}")
    print(f"    {'composite':15s}: {tm.get('composite_score', 0.0):.3f}")

METRICS:
  Composite Score: 0.503

  Key Metrics:
    accuracy       : 0.520  (weight=0.6, contribution=0.3120)
    f1             : N/A   (stage 3 metric — run test eval to compute)
    safety         : N/A   (stage 3 metric — run test eval to compute)
    robustness     : N/A   (stage 3 metric — run test eval to compute)
    efficiency     : N/A   (stage 3 metric — run test eval to compute)
  Screening Metrics:
    token_f1       : 0.478  (weight=0.4, contribution=0.1913)

  Test Set Metrics (Stage 3 — all 5 criteria):
    accuracy       : 0.473
    f1             : 0.451
    safety         : 0.930
    robustness     : 0.841
    efficiency     : 0.780
    composite      : 0.594


In [10]:
print(optimizer.visualize_optimization_trajectory())


OPTIMIZATION TRAJECTORY

Generation | Best Score | Overall Best | Improvement
------------------------------------------------------------
   1       | 0.503      | 0.503       | +0.068 █████████████████████████
   2       | 0.503      | 0.503       | +0.000 █████████████████████████
   3       | 0.503      | 0.503       | +0.000 █████████████████████████




In [11]:
ыreport = optimizer.get_optimization_report()

print("OPTIMIZATION REPORT:")
print("Overall Statistics:")
print(f"   Total time: {report['optimization_info']['total_time_seconds']:.2f}s")
print(f"   Generations: {report['optimization_info']['generations']}")
print(f"   Total nodes explored: {report['component_statistics']['history']['total_nodes']}")

print("Component Statistics:")
print(f"   Local optimizer iterations: {report['component_statistics']['local_optimizer']['total_iterations']}")
print(f"   Local improvements: {report['component_statistics']['local_optimizer']['improvements_count']}")
print(f"   Global optimizer steps: {report['component_statistics']['global_optimizer']['total_global_steps']}")
print(f"   Successful global changes: {report['component_statistics']['global_optimizer']['successful_global_changes']}")

print("Best Global Strategies:")
for i, strategy in enumerate(report['best_global_strategies'][:3], 1):
    print(f"   {i}. Score {strategy['score']:.3f}")
    print(f"      {strategy['strategy']['description'][:70]}...")


OPTIMIZATION REPORT:
Overall Statistics:
   Total time: 2090.44s
   Generations: 3
   Total nodes explored: 31
Component Statistics:
   Local optimizer iterations: 10
   Local improvements: 2
   Global optimizer steps: 1
   Successful global changes: 0
Best Global Strategies:
   1. Score 0.488
      meta-optimizer...
   2. Score 0.460
      meta-optimizer...
   3. Score 0.449
      meta-optimizer...


In [12]:
lineage = optimizer.history.get_lineage(best_node.id)

print("EVOLUTION OF BEST PROMPT:")
print("="*80)

for i, node in enumerate(lineage):
    print(f"Step {i}: Generation {node.generation}, Source: {node.source.value}")
    if node.is_evaluated:
        print(f"  Score: {node.metrics.composite_score():.3f}")

    if node.operations:
        print(f"  Operations:")
        for op in node.operations:
            print(f"    - {op.description}...")

    if i < len(lineage) - 1:  
        print("  ↓")


EVOLUTION OF BEST PROMPT:
Step 0: Generation 0, Source: initial
  Score: 0.435
  ↓
Step 1: Generation 1, Source: local
  Score: 0.503
  Operations:
    - Introduced a constraint to ensure the model generates answers solely based on factual details provided in the context....


In [13]:
print("FINAL SUMMARY")
print("="*80)

history_stats = optimizer.history.get_statistics()
local_stats = optimizer.local_optimizer.get_statistics()
global_stats = optimizer.global_optimizer.get_statistics()

print("Overall Statistics:")
print(f"  Total nodes explored: {history_stats['total_nodes']}")
print(f"  Evaluations performed: {history_stats['evaluated_nodes']}")
print(f"  Generations completed: {history_stats['max_generation']}")
print(f"  Best score achieved: {history_stats['best_score']:.3f}")
print(f"  Average score: {history_stats['avg_score']:.3f}")

print("Local Optimization:")
print(f"  Total iterations: {local_stats['total_iterations']}")
print(f"  Improvements found: {local_stats['improvements_count']}")
print(f"  Success rate: {local_stats['improvement_rate']:.1%}")

print("Global Optimization:")
print(f"  Total global steps: {global_stats['total_global_steps']}")
print(f"  Candidates generated: {global_stats['total_candidates_generated']}")
print(f"  Successful changes: {global_stats['successful_global_changes']}")
print(f"  Success rate: {global_stats['success_rate']:.1%}")

print("Optimization complete!")
print(f"   Results saved to: ./optimization_results/")

FINAL SUMMARY
Overall Statistics:
  Total nodes explored: 31
  Evaluations performed: 31
  Generations completed: 4
  Best score achieved: 0.503
  Average score: 0.442
Local Optimization:
  Total iterations: 10
  Improvements found: 2
  Success rate: 20.0%
Global Optimization:
  Total global steps: 1
  Candidates generated: 6
  Successful changes: 0
  Success rate: 0.0%
Optimization complete!
   Results saved to: ./optimization_results/
